## 1. Imports & Configuration

In [ ]:
import re
import warnings
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import (
    r2_score, mean_absolute_error,
    mean_squared_error, mean_absolute_percentage_error
)
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

# ── Reproducibility ──────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Config ───────────────────────────────────────────────────────────────────
CENTER_ID      = 13          # fulfillment center to model
TRAIN_WEEK_CUT = 110         # weeks <= this go to train, rest to test
DATA_PATH      = 'train_merged.csv'
ENCODER_PATH   = 'encoder.pkl'
MODEL_PATH     = 'xgb_food_demand_center13.pkl'

print("✅ Libraries loaded successfully")
print(f"   Target center : {CENTER_ID}")
print(f"   Train/test cut: week {TRAIN_WEEK_CUT}")


✅ Libraries loaded successfully
   Target center : 13
   Train/test cut: week 110


## 2. Data Loading & Basic Quality Check

In [ ]:
df = pd.read_csv('train_merged.csv')

print(f"Shape          : {df.shape}")
print(f"Columns        : {list(df.columns)}")
print(f"\nDate range (week): {df['week'].min()} → {df['week'].max()}")
print(f"Unique centers : {df['center_id'].nunique()}")
print(f"Unique meals   : {df['meal_id'].nunique()}")


Shape          : (456548, 15)
Columns        : ['id', 'week', 'center_id', 'meal_id', 'checkout_price', 'base_price', 'emailer_for_promotion', 'homepage_featured', 'num_orders', 'city_code', 'region_code', 'center_type', 'op_area', 'category', 'cuisine']

Date range (week): 1 → 145
Unique centers : 77
Unique meals   : 51


In [ ]:
# ── Quality Checks ───────────────────────────────────────────────────────────
print("=" * 45)
print(f"Duplicate rows : {df.duplicated().sum()}")
print(f"\nNull values:")
print(df.isnull().sum())
print("=" * 45)

df.head()


Duplicate rows : 0

Null values:
id                       0
week                     0
center_id                0
meal_id                  0
checkout_price           0
base_price               0
emailer_for_promotion    0
homepage_featured        0
num_orders               0
city_code                0
region_code              0
center_type              0
op_area                  0
category                 0
cuisine                  0
dtype: int64


,id,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,city_code,region_code,center_type,op_area,category,cuisine
0,1379560,1,55,1885,136.83,152.29,0,0,177,647,56,TYPE_C,2.00,Beverages,Thai
1,1466964,1,55,1993,136.83,135.83,0,0,270,647,56,TYPE_C,2.00,Beverages,Thai
2,1346989,1,55,2539,134.86,135.86,0,0,189,647,56,TYPE_C,2.00,Beverages,Thai
3,1338232,1,55,2139,339.50,437.53,0,0,54,647,56,TYPE_C,2.00,Beverages,Indian
4,1448490,1,55,2631,243.50,242.50,0,0,40,647,56,TYPE_C,2.00,Beverages,Indian


In [ ]:
df.describe()


,id,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,city_code,region_code,op_area
count,456548.00,456548.00,456548.00,456548.00,456548.00,456548.00,456548.00,456548.00,456548.00,456548.00,456548.00,456548.00
mean,1250096.31,74.77,82.11,2024.34,332.24,354.16,0.08,0.11,261.87,601.55,56.61,4.08
std,144354.82,41.52,45.98,547.42,152.94,160.72,0.27,0.31,395.92,66.20,17.64,1.09
min,1000000.00,1.00,10.00,1062.00,2.97,55.35,0.00,0.00,13.00,456.00,23.00,0.90
25%,1124998.75,39.00,43.00,1558.00,228.95,243.50,0.00,0.00,54.00,553.00,34.00,3.60
50%,1250183.50,76.00,76.00,1993.00,296.82,310.46,0.00,0.00,136.00,596.00,56.00,4.00
75%,1375140.25,111.00,110.00,2539.00,445.23,458.87,0.00,0.00,324.00,651.00,77.00,4.50
max,1499999.00,145.00,186.00,2956.00,866.27,866.27,1.00,1.00,24299.00,713.00,93.00,7.00
